<a href="https://colab.research.google.com/github/Monia1801/Prompt-Engineering/blob/main/Exp_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install langchain-text-splitters langchain-community pypdf faiss-cpu

In [3]:
!pip install langchain-google-genai

In [4]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from google.colab import userdata

/tmp/ipykernel_12913/2270064448.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [5]:
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
embeddings_model = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2")
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0.0)

In [7]:
loader = PyPDFLoader(file_path=input("Enter the pdf path: "))
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
document_chunks = text_splitter.split_documents(loader.load())
vector_db = FAISS.from_documents(document_chunks, embeddings_model)
retriever = vector_db.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 5}
 )

Enter the pdf path: /content/sample_resume.pdf


In [ ]:
while True:
  user_query = input("Enter the query related to pdf: ")
  if user_query == "exit":
    print("Shutting down RAG Engine.")
    break
  retrieved_docs = retriever.invoke(user_query)
  context_block = "\n---\n".join([doc.page_content for doc in retrieved_docs])
  rag_prompt = ChatPromptTemplate.from_messages([
      ("system", (
          "You are an elite research analyst. Answer the question using ONLY "
          "the provided context snippets. If you do not possess the "
          "information, state that clearly.\n\n"
          "Retrieved PDF Context:\n{context}"
        )),
        ("human", "{question}")
    ])
  final_output = (rag_prompt | llm).invoke({
      "context": context_block,
      "question": user_query
    })
  print(f"RAG Generation Output:\n{final_output.content[0]["text"]}")

Enter the query related to pdf: What are the skills of the candidate? 
RAG Generation Output:
Based on the provided document, the candidate's skills are as follows:

*   **Programming Languages:** Python, SQL, Scala
*   **Data Engineering:** ETL/ELT Pipelines, Data Modeling, Data Warehousing, Batch & Stream Processing
*   **Big Data Technologies:** Apache Spark, Hadoop, Kafka, Hive
*   **Databases:** PostgreSQL, MySQL, MongoDB, Snowflake, BigQuery
*   **Cloud Platforms:** 
    *   AWS (S3, Redshift, Glue, Lambda, ECS)
    *   GCP (BigQuery, Dataflow)
*   **Workflow Orchestration:** Apache Airflow, Prefect
*   **MLOps & DevOps:** Docker, GitHub
*   **Other:** Tableau (listed under the GitHub/LinkedIn section)
